Dataset definition

In [54]:
import pandas as pd

df = pd.read_csv("production_log.csv")

df.head(5)

,batch_id,date,shift,machine,units_produced,defective_units
0,B0001,2026-01-01,Morning,M1,437,NaN
1,B0002,2026-01-01,Evening,M2,454,NaN
2,B0003,2026-01-01,Night,M3,471,33.0
3,B0004,2026-01-01,Morning,M4,488,5.0
4,B0005,2026-01-01,Evening,M5,505,12.0


In [8]:
df.shape

(1000, 6)

In [9]:
df.columns


Index(['batch_id', 'date', 'shift', 'machine', 'units_produced',
       'defective_units'],
      dtype='str')

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   batch_id         1000 non-null   str    
 1   date             1000 non-null   str    
 2   shift            999 non-null    str    
 3   machine          999 non-null    str    
 4   units_produced   1000 non-null   int64  
 5   defective_units  999 non-null    float64
dtypes: float64(1), int64(1), str(4)
memory usage: 47.0 KB


In [17]:

df.isna().sum()

batch_id           0
date               0
shift              1
machine            1
units_produced     0
defective_units    1
dtype: int64

In [21]:
# The "machine" column shows which machine produced each batch in the log
df["machine"]



0      M1
1      M2
2      M3
3      M4
4      M5
       ..
995    M1
996    M2
997    M3
998    M4
999    M5
Name: machine, Length: 1000, dtype: str

In [23]:
# This expression does not create a completely independent DataFrame in memory.
# In pandas, df[["machine", "shift", "units_produced"]] returns a *view-like* subset
# (a new DataFrame object that still references the same underlying data buffer
# whenever possible). This means:
# - It is a DataFrame with only the selected columns.
# - It usually does NOT copy all the data: changes to the original df's column
#   values may be reflected here, and vice versa, depending on the operation.
# - To force an entirely separate DataFrame with its own memory, use .copy():
#       df_subset = df[["machine", "shift", "units_produced"]].copy()
df[["machine", "shift", "units_produced"]]



,machine,shift,units_produced
0,M1,Morning,437
1,M2,Evening,454
2,M3,Night,471
3,M4,Morning,488
4,M5,Evening,505
...,...,...,...
995,M1,Night,432
996,M2,Morning,449
997,M3,Evening,466
998,M4,Night,483


In [26]:

df[df["machine"] == "M1"]


,batch_id,date,shift,machine,units_produced,defective_units
0,B0001,2026-01-01,Morning,M1,437,12.0
5,B0006,2026-01-01,Night,M1,522,22.0
10,B0011,2026-01-01,Evening,M1,427,26.0
15,B0016,2026-01-02,Morning,M1,512,5.0
20,B0021,2026-01-02,Night,M1,597,15.0
...,...,...,...,...,...,...
975,B0976,2026-03-07,Morning,M1,452,5.0
980,B0981,2026-03-07,Night,M1,537,15.0
985,B0986,2026-03-07,Evening,M1,442,19.0
990,B0991,2026-03-08,Morning,M1,527,26.0


In [44]:

df[df["defective_units"].isna()]


,batch_id,date,shift,machine,units_produced,defective_units
0,B0001,2026-01-01,Morning,M1,437,NaN
1,B0002,2026-01-01,Evening,M2,454,NaN
252,B0253,2026-01-17,Morning,M3,581,NaN


In [48]:
df.sort_values("defective_units", ascending=False).head(5)

,batch_id,date,shift,machine,units_produced,defective_units
2,B0003,2026-01-01,Night,M3,471,33.0
662,B0663,2026-02-14,Night,M3,531,33.0
542,B0543,2026-02-06,Night,M3,471,33.0
62,B0063,2026-01-05,Night,M3,591,33.0
362,B0363,2026-01-25,Night,M3,471,33.0


In [ ]:
kpi_series = df["defective_units"] / df["units_produced"]

df["scrap_rate"] = kpi_series


# This line is doing three main things in a row (that’s what “method chaining” means):
#
# 1. `df[df["scrap_rate"] > 0.05]`
#    - Looks at the `scrap_rate` column of the DataFrame `df`.
#    - Keeps only the rows where `scrap_rate` is greater than `0.05`.
#    - The result is a smaller DataFrame containing only “high scrap rate” rows.
#
# 2. `.groupby("machine")`
#    - Takes that filtered DataFrame and groups the rows by the value in the `machine` column.
#    - So all rows for the same machine (M1, M2, etc.) are collected into their own group.
#
# 3. `["scrap_rate"].describe()`
#    - From each machine group, it selects only the `scrap_rate` column.
#    - Then `describe()` calculates summary statistics for `scrap_rate` within each machine group:
#      - count (how many rows)
#      - mean (average scrap rate)
#      - std (standard deviation)
#      - min (minimum scrap rate)
#      - 25% / 50% / 75% (quartiles, including median)
#      - max (maximum scrap rate)
#
# Overall, this code:
# - Filters to only rows where scrap rate is above 5%,
# - Groups those rows by machine,
# - And then shows a table of summary statistics of scrap rate for each machine.

df[df["scrap_rate"] > 0.05].groupby("machine")["scrap_rate"].describe()


,batch_id,date,shift,machine,units_produced,defective_units,scrap_rate
0,B0001,2026-01-01,Morning,M1,437,NaN,NaN
1,B0002,2026-01-01,Evening,M2,454,NaN,NaN
2,B0003,2026-01-01,Night,M3,471,33.0,0.070064
3,B0004,2026-01-01,Morning,M4,488,5.0,0.010246
4,B0005,2026-01-01,Evening,M5,505,12.0,0.023762
